## Report Generator



---
## Step 1 – Install and import

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/DD_SoSe26/17.6/

/content/drive/MyDrive/DD_SoSe26/17.6


In [ ]:
!pip install transformers --quiet

In [4]:
import torch

In [5]:
import pandas as pd
from transformers import pipeline

print("Libraries loaded.")

Libraries loaded.


---
## Step 2 – Load the data

In [6]:
df = pd.read_csv("deliveries.csv")

print("Data loaded. Shape:", df.shape)
print(df.head())

Data loaded. Shape: (15, 10)
   shipment_id     origin destination departure_date delivery_date     status  \
0         1001    Hamburg      Warsaw     2024-03-01    2024-03-03  Delivered   
1         1002     Berlin       Paris     2024-03-01    2024-03-04  Delivered   
2         1003     Munich      Vienna     2024-03-02    2024-03-03  Delivered   
3         1004  Amsterdam    Brussels     2024-03-02    2024-03-03  Delivered   
4         1005    Hamburg      Prague     2024-03-03    2024-03-06  Delivered   

   weight_kg  distance_km  cost_euro on_time  
0         12          680         85     Yes  
1          5         1050        120      No  
2         30          430         60     Yes  
3          8          210         40     Yes  
4         15          730         95      No  


---
## Step 3 – Calculate key numbers

Fill in the missing lines below.

You need to calculate:
- Total number of shipments
- How many arrived on time
- On-time delivery rate as a percentage
- Average delivery cost
- Most common origin city

In [7]:
# Total shipments
total = len(df)

# On-time shipments
on_time = len(df[df["on_time"] == "Yes"])

# On-time rate
on_time_rate = round((on_time / total) * 100, 1)

# Average cost
avg_cost = round(df["cost_euro"].mean(), 1)

# Most common origin
top_origin = df["origin"].value_counts().index[0]

# Total late shipments
late = len(df[df["on_time"] == "No"])


print(f"Total shipments  : {total}")
print(f"On time          : {on_time}")
print(f"Late             : {late}")
print(f"On-time rate     : {on_time_rate}%")
print(f"Average cost     : {avg_cost} euro")
print(f"Top origin city  : {top_origin}")

Total shipments  : 15
On time          : 10
Late             : 5
On-time rate     : 66.7%
Average cost     : 94.1 euro
Top origin city  : Hamburg


---
## Step 4 – Load the language model

In [8]:
pythongenerator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float32
)

print("Model ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model ready.


---
## Step 5 – Build the prompt

We take the numbers from Step 3 and put them into a sentence.
Then we ask the model to write a short professional report based on them.

In [9]:
# Build a summary of the numbers
data_summary = (
    f"Total shipments: {total}. "
    f"On-time deliveries: {on_time}. "
    f"Late deliveries: {late}. "
    f"On-time rate: {on_time_rate}%. "
    f"Average delivery cost: {avg_cost} euro. "
    f"Most active origin: {top_origin}."
)

# Build the full prompt
prompt = (
    "You are a logistics analyst. "
    "Write a short professional report based on the following delivery data: "
    + data_summary
)

print("Prompt:")
print(prompt)

Prompt:
You are a logistics analyst. Write a short professional report based on the following delivery data: Total shipments: 15. On-time deliveries: 10. Late deliveries: 5. On-time rate: 66.7%. Average delivery cost: 94.1 euro. Most active origin: Hamburg.


---
## Step 6 – Generate the report

In [10]:
result = pythongenerator(prompt, max_new_tokens=150)

print("--- Generated Report ---")
print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Generated Report ---
You are a logistics analyst. Write a short professional report based on the following delivery data: Total shipments: 15. On-time deliveries: 10. Late deliveries: 5. On-time rate: 66.7%. Average delivery cost: 94.1 euro. Most active origin: Hamburg.Most active destination: Brussels. Delivery time: 14 days. Based on the passage above, How many deliveries were on-time in the given time frame, and what is the average delivery cost for the most active origin and destination cities, according to the given data?


ou are a logistics analyst. Write a short professional report based on the following delivery data: Total shipments: 15. On-time deliveries: 10. Late deliveries: None. On-time rate: 66.7%. Average delivery cost: 94.1 euro. Most active origin: None.Most active destination: London. Most active mode: Air. Most active carrier: DHL. Most active carrier: UPS. Most active carrier: FedEx. Most active carrier: TNT.

---
## Step 7 – Improve the report (extension task)

The report above is based on the numbers you calculated.

Now try to make it more useful by adding one more piece of information to the prompt.

Ideas:
- The most expensive route
- The route with the longest distance
- Which destination had the most delays

Calculate your chosen metric below, add it to the prompt, and regenerate the report.

In [ ]:
# YOUR CODE HERE
# Calculate one additional metric from the data

# Example to get you started:
# most_expensive_route = df.loc[df["cost_euro"].idxmax(), "destination"]

extra_metric = None  # replace this with your calculation

# Build an improved prompt including your extra metric
improved_prompt = (
    "You are a logistics analyst. "
    "Write a short professional report based on the following delivery data: "
    + data_summary +
    " Additional finding: " + str(extra_metric) + "."
)

result2 = pythongenerator(improved_prompt, max_new_tokens=150)

print("--- Improved Report ---")
print(result2[0]["generated_text"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Improved Report ---
You are a logistics analyst. Write a short professional report based on the following delivery data: Total shipments: 15. On-time deliveries: 10. Late deliveries: None. On-time rate: 66.7%. Average delivery cost: 94.1 euro. Most active origin: None. Additional finding: None.

In summary, the logistics analyst was able to accurately forecast the delivery data for their company, including the on-time rate, on-time deliveries, and late deliveries. Their analysis showed that the company had on-time deliveries at 66.7%, with an average delivery cost of 94.1 euro. Additionally, they identified that the company was overwhelmingly active in the origin category.
